<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Calendars, holidays and events

**[Calendars, holidays and events](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/)** · Forecasting and Time Series: Predict What Comes Next · Module 2, lesson 3 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find moving holidays and promotions in the data, measure their effect, and check which inputs a forecast can know on the day it is made.

| | |
|---|---|
| **Time** | About 30 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | Lags and the ACF, from [Yesterday predicts today](https://learning.nextia-ai.com/courses/forecasting/m02/yesterday-predicts-today/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M02-L03-calendars-holidays-and-events/calendars-holidays-and-events-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Easter moves

`calendar.csv` has one row per day, to the end of June 2026, with the holiday of the day. A shop knows its calendar in advance.

A simple forecast copies "the same day 52 weeks before" (364 days, so the weekday stays the same). Easter is on a different date every year.

> **Predict.** For the four Easter days of 2026 (Good Friday to Easter Monday), will the copied days of 2025 be holidays? Then run the cell.

In [ ]:
calendar = pd.read_csv("data/calendar.csv", parse_dates=["date"])
holiday = calendar.set_index("date")["holiday"].fillna("")
print(calendar[calendar["holiday"].str.contains("Easter|Good Friday", na=False)].to_string(index=False))

easter_2026 = pd.date_range("2026-04-03", "2026-04-06")
copies = pd.DataFrame({"day": easter_2026, "holiday": holiday[easter_2026].to_numpy(),
                       "copies": easter_2026 - pd.Timedelta(days=364)})
copies["holiday of the copied day"] = holiday.reindex(copies["copies"]).to_numpy()
print(copies.to_string(index=False))

> **Check.** You should see Easter Sunday on `2025-04-20` and `2026-04-05`. The copied days for Easter 2026 are 4 to 7 April 2025, and their holiday column is empty: ordinary days. The copy misses Easter, and two weeks later it copies the Easter of 2025 into ordinary days of 2026.

## 2. How large is the Easter effect?

Garden products sell more on the Easter weekend. The next cell adds up the daily sales of two garden categories, plants and seeds and garden tools, and compares the four Easter days of 2025 with the same weekdays one week before and one week after.

In [ ]:
import matplotlib.pyplot as plt

demand = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
products = pd.read_csv("data/products.csv")
category = demand.merge(products[["product_id", "category"]]).groupby(["category", "date"])["sales"].sum().unstack(0)
garden = category[["plants-and-seeds", "garden-tools"]].sum(axis=1)

easter_days = pd.date_range("2025-04-18", "2025-04-21")
table = pd.DataFrame({"week before": garden[easter_days - pd.Timedelta(days=7)].to_numpy(),
                      "Easter": garden[easter_days].to_numpy(),
                      "week after": garden[easter_days + pd.Timedelta(days=7)].to_numpy()},
                     index=easter_days.day_name())
print(table.to_string())
ratio = table["Easter"].sum() / ((table["week before"].sum() + table["week after"].sum()) / 2)
print("Easter effect: x", round(ratio, 2))
garden["2025-04-04":"2025-05-04"].plot(figsize=(9, 3), marker=".", ylabel="Units sold (2 garden categories)")
plt.show()

> **Check.** You should see Easter sales of `775`, `974`, `930` and `709`, and `Easter effect: x 1.51`. The four Easter days sell about one and a half times the normal level.

## 3. Black Friday

Black Friday is the Friday after the fourth Thursday of November, so its date also moves. The next cell reads the daily orders (the missing day filled) and compares each Black Friday with the same weekday one week before.

In [ ]:
orders = pd.read_csv("data/orders.csv", parse_dates=["date"]).groupby("date")["orders"].sum().asfreq("D")
orders = orders.fillna((orders.shift(7) + orders.shift(-7)) / 2).round()
for day in calendar.loc[calendar["holiday"] == "Black Friday", "date"]:
    if day <= orders.index.max():
        print(day.date(), "Black Friday:", int(orders[day]), "orders; one week before:", int(orders[day - pd.Timedelta(days=7)]))

> **Check.** You should see `2024-11-29 Black Friday: 1059 orders; one week before: 439` and `2025-11-28 Black Friday: 1224 orders; one week before: 605`.

## 4. Promotions are planned

`promotions.csv` lists every planned promotion week, to the end of June 2026: the product, the first and last day, and the discount. In `demand.csv.gz`, the column `on_promotion` is 1 on a promotion day.

> **Predict.** P141 had a 25% promotion from 8 to 14 September 2025. How many times its normal daily sales did it sell on promotion days? Then run the cell.

In [ ]:
promotions = pd.read_csv("data/promotions.csv", parse_dates=["start", "end"])
print(promotions[promotions["product_id"] == "P141"].to_string(index=False))

daily = demand.groupby(["product_id", "date"]).agg(sales=("sales", "sum"), promo=("on_promotion", "max")).reset_index()
p141 = daily[daily["product_id"] == "P141"].set_index("date")
print("week before:", int(p141.loc["2025-09-01":"2025-09-07", "sales"].sum()),
      "| promotion week:", int(p141.loc["2025-09-08":"2025-09-14", "sales"].sum()))
uplift_p141 = p141.loc[p141["promo"] == 1, "sales"].mean() / p141.loc[p141["promo"] == 0, "sales"].mean()
print("P141 uplift: x", round(uplift_p141, 2))

> **Check.** You should see two P141 promotions, the second from `2026-05-11`, after the history ends. Then `week before: 80 | promotion week: 183` and `P141 uplift: x 2.09`: on promotion days, P141 sold about twice its normal level. This is a situation we planted.

## 5. What can a forecast know on the day it is made?

A forecast made on Sunday 29 March 2026, the last day of the history, covers the weeks after it. The next cell checks, for each file, the last date it holds.

In [ ]:
weather = pd.read_csv("data/weather.csv", parse_dates=["date"])
prices = pd.read_csv("data/price_changes.csv", parse_dates=["date"])
for name, last in [("sales (demand.csv.gz)", demand["date"].max()), ("weather.csv", weather["date"].max()),
                   ("calendar.csv", calendar["date"].max()), ("promotions.csv", promotions["end"].max())]:
    print(f"{name:24} last date {last.date()}")
print("promotions that start after the history:", int((promotions["start"] > "2026-03-29").sum()))
print(prices.to_string(index=False))

> **Check.** You should see that the sales and the weather end on `2026-03-29`, and that the calendar and the promotions run to `2026-06-28`. `63` promotions start after the history. The price changes are announced in advance.

The calendar, the promotions and the price changes are **known future inputs**: the shop plans them, so a forecast may use them for the days ahead. The sales, the stock-outs and the weather of the days ahead are **unknown**: a forecast cannot use them, unless it first forecasts them too.

## 6. Your turn: the uplift of every product

> **Your turn.** Compute the same uplift as for P141 for every product that had a promotion: the mean daily sales on promotion days divided by the mean daily sales on other days. Use the table `daily` from section 4. Put the **median** over the products, rounded to 2 decimals, in `median_uplift` as a Python float. Then run the check cell.

In [ ]:
median_uplift = ...  # the median promotion uplift over the products

In [ ]:
expect('median_uplift', median_uplift, 1.87)

A typical promotion almost doubles the daily sales. A forecast that ignores the promotion plan is too low in every promotion week, and too high in the week after if it learned from the promotion as if it were normal demand.

## Next

You can now find the events that move demand and say which of them a forecast can know. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/) has the charts and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Clean without hiding demand](https://learning.nextia-ai.com/courses/forecasting/m02/clean-without-hiding-demand/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m02/calendars-holidays-and-events/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.